In [ ]:
"""
A script to fetch the loci in a search query from the ANTARES API.
"""

#from antares_client.search import search

QUERY = {
    "query": {
        "bool": {
            "filter": {
                "bool": {
                    "must": [
                        {
                            "terms": {
                                "tags": [
                                    "lantern_xgboost_t2.0.7_c0.95"
                                ]
                            }
                        },
                        {
                            "exists": {
                                "field": "properties.survey.lsst"
                            }
                        }
                    ]
                }
            }
        }
    }
}


#def process_locus(locus):
#    """Put your custom processing logic here."""
#    print(locus.locus_id)


#def main():
#    for locus in search(QUERY):
#        process_locus(locus)


#if __name__ == "__main__":
#    main()



In [ ]:
import json
import pandas as pd

from pathlib import Path
from itertools import islice
from concurrent.futures import ThreadPoolExecutor, as_completed

from antares_client.search import search

In [ ]:
# ============================================================
# Settings
# ============================================================

outdir = Path("antares_data")

locus_dir = outdir / "loci"
alert_dir = outdir / "alerts"

locus_dir.mkdir(parents=True, exist_ok=True)
alert_dir.mkdir(parents=True, exist_ok=True)

# Memory / disk batching
alert_batch_size = 100_000
locus_batch_size = 5_000

# Network concurrency
max_workers = 4

# Number of loci submitted at a time
fetch_batch_size = 32

# Test with only the first 1000 loci
max_loci = 1000

In [ ]:
# ============================================================
# Process one locus
# ============================================================

def process_locus(n_loci, locus):

    # ----------------------------
    # Locus-level information
    # ----------------------------

    locus_row = {
        "locus_id": locus.locus_id,
        "ra": locus.ra,
        "dec": locus.dec,
        "tags": locus.tags,
        "locus_properties": json.dumps(
            locus.properties,
            separators=(",", ":"),
            default=str,
        ),
    }

    # ----------------------------
    # Alert-level information
    # ----------------------------

    alert_rows_local = []

    n_alerts_total_local = 0
    n_alerts_saved_local = 0
    n_non_lsst_skipped_local = 0

    for alert in locus.alerts:

        n_alerts_total_local += 1

        # Keep only LSST alerts
        if not alert.alert_id.startswith("lsst:"):
            n_non_lsst_skipped_local += 1
            continue

        alert_rows_local.append({
            "locus_id": locus.locus_id,
            "alert_id": alert.alert_id,
            "mjd": alert.mjd,
            "alert_properties": json.dumps(
                alert.properties,
                separators=(",", ":"),
                default=str,
            ),
        })

        n_alerts_saved_local += 1

    return (
        n_loci,
        locus_row,
        alert_rows_local,
        n_alerts_total_local,
        n_alerts_saved_local,
        n_non_lsst_skipped_local,
    )


# ============================================================
# Temporary storage
# ============================================================

locus_rows = []
alert_rows = []

locus_part = 0
alert_part = 0

n_alerts_total = 0
n_alerts_saved = 0
n_non_lsst_skipped = 0
n_loci_processed = 0


# ============================================================
# ANTARES search iterator
# Only take the first 1000 loci
# ============================================================

locus_iterator = enumerate(
    islice(search(QUERY), max_loci),
    start=1,
)

In [ ]:
%%time

# ============================================================
# Parallel downloading
# ============================================================

with ThreadPoolExecutor(max_workers=max_workers) as executor:

    while True:

        # Take only a small batch from search()
        batch = list(
            islice(
                locus_iterator,
                fetch_batch_size,
            )
        )

        if not batch:
            break

        # Download alerts concurrently
        futures = [
            executor.submit(
                process_locus,
                n_loci,
                locus,
            )
            for n_loci, locus in batch
        ]

        # Collect finished loci
        for future in as_completed(futures):

            (
                n_loci,
                locus_row,
                alert_rows_local,
                n_total_local,
                n_saved_local,
                n_non_lsst_local,
            ) = future.result()

            n_loci_processed += 1

            locus_rows.append(locus_row)
            alert_rows.extend(alert_rows_local)

            n_alerts_total += n_total_local
            n_alerts_saved += n_saved_local
            n_non_lsst_skipped += n_non_lsst_local


            # ======================================
            # Flush alerts
            # ======================================

            if len(alert_rows) >= alert_batch_size:

                pd.DataFrame(alert_rows).to_parquet(
                    alert_dir / f"alerts_{alert_part:05d}.parquet",
                    index=False,
                    compression="zstd",
                )

                print(
                    f"Saved alert part {alert_part:05d}: "
                    f"{len(alert_rows):,} alerts"
                )

                alert_rows = []
                alert_part += 1


            # ======================================
            # Flush loci
            # ======================================

            if len(locus_rows) >= locus_batch_size:

                pd.DataFrame(locus_rows).to_parquet(
                    locus_dir / f"loci_{locus_part:05d}.parquet",
                    index=False,
                    compression="zstd",
                )

                print(
                    f"Saved locus part {locus_part:05d}: "
                    f"{len(locus_rows):,} loci"
                )

                locus_rows = []
                locus_part += 1


        print(
            f"Processed {n_loci_processed:,} loci total | "
            f"LSST alerts saved total: {n_alerts_saved:,} | "
            f"non-LSST alerts skipped total: {n_non_lsst_skipped:,}"
            )


# ============================================================
# Save remaining alerts
# ============================================================

if alert_rows:

    pd.DataFrame(alert_rows).to_parquet(
        alert_dir / f"alerts_{alert_part:05d}.parquet",
        index=False,
        compression="zstd",
    )

    print(
        f"Saved final alert part {alert_part:05d}: "
        f"{len(alert_rows):,} alerts"
    )


# ============================================================
# Save remaining loci
# ============================================================

if locus_rows:

    pd.DataFrame(locus_rows).to_parquet(
        locus_dir / f"loci_{locus_part:05d}.parquet",
        index=False,
        compression="zstd",
    )

    print(
        f"Saved final locus part {locus_part:05d}: "
        f"{len(locus_rows):,} loci"
    )


# ============================================================
# Summary
# ============================================================

print()
print("Finished")
print(f"Loci processed:          {n_loci_processed:,}")
print(f"All alerts encountered:  {n_alerts_total:,}")
print(f"LSST alerts saved:       {n_alerts_saved:,}")
print(f"Non-LSST alerts skipped: {n_non_lsst_skipped:,}")

In [ ]:
import pyarrow.dataset as ds

loci_ds = ds.dataset(
    "antares_data/loci",
    format="parquet",
)

alerts_ds = ds.dataset(
    "antares_data/alerts",
    format="parquet",
)

In [ ]:
import pyarrow.parquet as pq

table = pq.read_table(
    "antares_data/alerts/alerts_00000.parquet"
)

print(table.schema)
print(table.num_rows)

In [ ]:
locus_id = "ANT2020etnce"

locus = loci_ds.to_table(
    filter=ds.field("locus_id") == locus_id
).to_pandas()

alerts = alerts_ds.to_table(
    filter=ds.field("locus_id") == locus_id
).to_pandas()

In [ ]:
locus

In [ ]:
alerts

In [ ]:
l = alerts['alert_id'].to_list()
l.sort()
print(l[:5])

In [ ]:
coord = loci_ds.to_table(
    columns=["locus_id", "ra", "dec"],
    filter=ds.field("locus_id") == locus_id,
).to_pandas()

In [ ]:
coord